<a href="https://colab.research.google.com/github/iresiragusa/NLP-EN-JP/blob/main/E2_instruct_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Inference with Instruct LLMs: chat templates, few-shot learning, and CoT (Chain of Thought)
======================================================


This notebook analyzes the use of instruct-type models, the use of chat templates, and two types of prompting techniques (few-shot and CoT) with [Qwen 3 4B](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507) model.

Install and imports
============================================================================

To execute this notebook are required the following libraries, already installed in google colab

```pip install transformers torch```

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig
import warnings
warnings.filterwarnings('ignore')

In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# --- Qwen 3.2 Instruct ---
print("Loading Qwen 3.2 4B Instruct...")
model_name = "Qwen/Qwen3-4B-Instruct-2507"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map=device,
    trust_remote_code=True
)


Loading Qwen 3.2 4B Instruct...


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

In [3]:
# definine a function to infer the LLM

def ask(message, tokenizer, model, device):

    # Tokenization
    inputs = tokenizer(
        message,
        return_tensors="pt"
    )
    inputs['input_ids'] = inputs['input_ids'].to(device)
    inputs['attention_mask'] = inputs['attention_mask'].to(device)

    # generation configs
    gen_config = GenerationConfig(
        do_sample=True,
        max_new_tokens=256,
        temperature=0.7,
    )

    response = model.generate(
        **inputs,
        generation_config=gen_config)

    answer_full = tokenizer.decode(response[0], skip_special_tokens=False)
    answer = tokenizer.decode(response[0][len(inputs['input_ids'][0]):], skip_special_tokens=True)

    return answer_full, answer

`Chat template` These are standardized formats that Instruct models expect.
Each model has its own format stored within the tokenizer which specifies how different roles are managed.

- `system`
   
    - Defines the behavior and characteristics of the model
    - Sets the general context
    - Defines personality, style, and limitations
    - e.g. "You are a helpful and precise assistant"

- `user`
    - Represents the user's messages, such as questions or requests
    - Input that requires a response

- `assistant`
    - Represents the model's responses
    - Used for previous model's replies
    - During inference, the model generates this role

## Inferences with a message containing the system role

In [4]:
# Create a message with system role
messages_system = [
    {"role": "system", "content": "You are an expert geography assistant who responds concisely and precisely."},
    {"role": "user", "content": "What is the capital of Italy?"}
]

print("Input messages:")
print(messages_system)

# Apply template
prompt_system = tokenizer.apply_chat_template(
    messages_system,
    tokenize=False,             # not tokenize, shows only raw-textual tokens
    add_generation_prompt=True  # adds prompt for generation (assistant's role start)
)
print(prompt_system)

Input messages:
[{'role': 'system', 'content': 'You are an expert geography assistant who responds concisely and precisely.'}, {'role': 'user', 'content': 'What is the capital of Italy?'}]
<|im_start|>system
You are an expert geography assistant who responds concisely and precisely.<|im_end|>
<|im_start|>user
What is the capital of Italy?<|im_end|>
<|im_start|>assistant



In [5]:
answer_full, answer = ask(prompt_system, tokenizer, model, device)

print(f'\n\n*** Generated answer -> {answer}')



*** Generated answer -> The capital of Italy is Rome.


## Few-shot prompting

A prompting technique in which one or more examples are provided alongside the instruction.

Useful for helping the model learn the task and the output structure.

In [6]:
# Create messages for few-shot prompting
few_shot_messages = [
    {"role": "system", "content": "You are an Italian-to-English translator. Respond only with the translation."},
    {"role": "user", "content": "Potresti dirmi a che ora parte il prossimo treno per Roma?"},
    {"role": "assistant", "content": "Could you tell me what time the next train to Rome leaves"},
    {"role": "user", "content": "Stiamo ascoltando un vecchio disco."},
    {"role": "assistant", "content": "We are listening to an old record."},
    {"role": "user", "content": "Ieri ho studiato per l'esame di venerdì, spero di passarlo."}
    # expected answer -> Yesterday I studied for Friday exam, I hope that I will pass it.
]

print("Input messages:")
print(few_shot_messages)

prompt_few_shot = tokenizer.apply_chat_template(
    few_shot_messages,
    tokenize=False,
    add_generation_prompt=True
)
print(prompt_few_shot)

answer_full, answer = ask(prompt_few_shot, tokenizer, model, device)

print(f'\n\n*** Generated answer -> {answer}')

Input messages:
[{'role': 'system', 'content': 'You are an Italian-to-English translator. Respond only with the translation.'}, {'role': 'user', 'content': 'Potresti dirmi a che ora parte il prossimo treno per Roma?'}, {'role': 'assistant', 'content': 'Could you tell me what time the next train to Rome leaves'}, {'role': 'user', 'content': 'Stiamo ascoltando un vecchio disco.'}, {'role': 'assistant', 'content': 'We are listening to an old record.'}, {'role': 'user', 'content': "Ieri ho studiato per l'esame di venerdì, spero di passarlo."}]
<|im_start|>system
You are an Italian-to-English translator. Respond only with the translation.<|im_end|>
<|im_start|>user
Potresti dirmi a che ora parte il prossimo treno per Roma?<|im_end|>
<|im_start|>assistant
Could you tell me what time the next train to Rome leaves<|im_end|>
<|im_start|>user
Stiamo ascoltando un vecchio disco.<|im_end|>
<|im_start|>assistant
We are listening to an old record.<|im_end|>
<|im_start|>user
Ieri ho studiato per l'es

## Chain of Thoughts

A prompting technique in which the model is asked to *reason* and return both its prediction and the logical steps necessary to arrive at that prediction.

Useful for the model when handling more complex tasks.

In [7]:
# Let's create a message for the CoT; we will ask the model to classify a text from the BBC-text dataset.
# Multiclass dataset of BBC news divided in 5 categories with reference to their topic.
# [Dataset reference](https://www.kaggle.com/datasets/yufengdev/bbc-fulltext-and-category/code)

instruction = """You are a text classifier. Follow these steps:\n
1. Identify key topics and entities in the article\n
2. Consider which category best matches these topics\n
3. Provide the category: business, politics, tech, sport, or entertainment\n

Format your response as:\n
Reasoning: [your analysis]\n
Category: [category name]\n"""

text = "ocean s twelve raids box office ocean s twelve  the crime caper sequel starring george clooney  brad pitt and julia roberts  has gone straight to number one in the us box office chart.  it took $40.8m (£21m) in weekend ticket sales  according to studio estimates. the sequel follows the master criminals as they try to pull off three major heists across europe. it knocked last week s number one  national treasure  into third place. wesley snipes  blade: trinity was in second  taking $16.1m (£8.4m). rounding out the top five was animated fable the polar express  starring tom hanks  and festive comedy christmas with the kranks.  ocean s twelve box office triumph marks the fourth-biggest opening for a december release in the us  after the three films in the lord of the rings trilogy. the sequel narrowly beat its 2001 predecessor  ocean s eleven which took $38.1m (£19.8m) on its opening weekend and $184m (£95.8m) in total. a remake of the 1960s film  starring frank sinatra and the rat pack  ocean s eleven was directed by oscar-winning director steven soderbergh. soderbergh returns to direct the hit sequel which reunites clooney  pitt and roberts with matt damon  andy garcia and elliott gould. catherine zeta-jones joins the all-star cast.  it s just a fun  good holiday movie   said dan fellman  president of distribution at warner bros. however  us critics were less complimentary about the $110m (£57.2m) project  with the los angeles times labelling it a  dispiriting vanity project . a milder review in the new york times dubbed the sequel  unabashedly trivial ."
# entertainment is the golden label

cot_messages = [
    {"role": "system", "content": instruction},
    {"role": "user", "content": text},
]

print("Input message:")
print(cot_messages)

prompt_cot = tokenizer.apply_chat_template(
    cot_messages,
    tokenize=False,
    add_generation_prompt=True
)
print(prompt_cot)

answer_full, answer = ask(prompt_cot, tokenizer, model, device)

print(f'\n\n*** Generated Answer -> {answer}')

Input message:
[{'role': 'system', 'content': 'You are a text classifier. Follow these steps:\n\n1. Identify key topics and entities in the article\n\n2. Consider which category best matches these topics\n\n3. Provide the category: business, politics, tech, sport, or entertainment\n\n\nFormat your response as:\n\nReasoning: [your analysis]\n\nCategory: [category name]\n'}, {'role': 'user', 'content': 'ocean s twelve raids box office ocean s twelve  the crime caper sequel starring george clooney  brad pitt and julia roberts  has gone straight to number one in the us box office chart.  it took $40.8m (£21m) in weekend ticket sales  according to studio estimates. the sequel follows the master criminals as they try to pull off three major heists across europe. it knocked last week s number one  national treasure  into third place. wesley snipes  blade: trinity was in second  taking $16.1m (£8.4m). rounding out the top five was animated fable the polar express  starring tom hanks  and festi